# Twitter Bot (X API) - Google Colab Edition

This notebook is a self-contained Twitter/X bot you can run straight from Colab, no separate files or project folder needed - just run the cells top to bottom.

**What this actually does:** it logs into your Twitter/X account using your API keys, then automatically posts tweets from a list you write, spaced out on a schedule (say, one every few hours) instead of you having to sit there and post them by hand. It also keeps track of what it's already posted so it doesn't repeat itself, and has an optional auto-reply feature that checks your mentions and responds to people automatically.

Think of it like a simple social media scheduler - the kind of thing a small account or a project might use to keep posting content on autopilot.

**Before you start, you'll need:**
1. A Twitter/X Developer account - apply at [developer.twitter.com](https://developer.twitter.com)
2. A Project + App created in the developer portal, with **Read and Write** permissions turned on
3. Four credentials from that app: API Key, API Key Secret, Access Token, Access Token Secret

**A couple of things worth knowing upfront:**
- Twitter/X's free API tier is very limited on posting (and at times has required a paid tier to post at all) - check your current access level in the developer portal before relying on this for anything important.
- Colab sessions disconnect after a while (usually a few hours of inactivity, or ~12 hours max). This notebook has a **dry-run mode** so you can test all the logic without needing real API keys or actually posting anything, and a Google Drive option so your posting history survives between sessions.


## 1. Install what we need

In [3]:
!pip install -q tweepy schedule

## 2. Imports

In [4]:
import json
import os
import random
import time
from datetime import datetime, timedelta

import tweepy


## 3. Dry-run switch

If `DRY_RUN = True`, nothing actually gets sent to Twitter - the bot just prints what it *would* post. This is handy for testing the scheduling and queue logic before you plug in real credentials, or if you don't have API access yet.

Set it to `False` once you're ready to post for real.

In [5]:
DRY_RUN = True  # <-- flip to False when you're ready to actually post


## 4. Enter your API credentials

This uses `getpass` so your keys don't get typed out in plain view or saved in the notebook file - each field asks you to paste the value in and hides it as you type. Skip this cell entirely if `DRY_RUN = True` and you just want to test the logic first.

In [6]:
from getpass import getpass

if not DRY_RUN:
    API_KEY = getpass("API Key: ")
    API_KEY_SECRET = getpass("API Key Secret: ")
    ACCESS_TOKEN = getpass("Access Token: ")
    ACCESS_TOKEN_SECRET = getpass("Access Token Secret: ")
else:
    print("DRY_RUN is on - skipping credential entry.")


DRY_RUN is on - skipping credential entry.


## 5. Connect to the API and confirm it works

`tweepy.Client` is the v2 API client. We authenticate with all four keys (this is called "user context" auth) since posting tweets needs write access tied to your specific account, not just read-only app access.

In [7]:
client = None

if not DRY_RUN:
    client = tweepy.Client(
        consumer_key=API_KEY,
        consumer_secret=API_KEY_SECRET,
        access_token=ACCESS_TOKEN,
        access_token_secret=ACCESS_TOKEN_SECRET,
    )

    try:
        me = client.get_me().data
        print(f"Connected successfully as @{me.username}")
    except tweepy.TweepyException as e:
        print(f"Couldn't authenticate - double check your keys and app permissions.\n{e}")
else:
    print("DRY_RUN is on - skipping the real connection check.")


DRY_RUN is on - skipping the real connection check.


## 6. (Optional) Save state to Google Drive

The bot keeps a small log file of what it's already posted, so it doesn't repeat tweets if you re-run the notebook or the session restarts. If you mount Google Drive, that log (and the tweet queue) can live there instead of disappearing when the Colab session ends.

Skip this cell if you're fine with the log resetting each session.

In [8]:
USE_DRIVE = False  # set True to persist the queue/log across sessions

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    STORAGE_DIR = "/content/drive/MyDrive/twitter_bot"
else:
    STORAGE_DIR = "/content/twitter_bot_data"

os.makedirs(STORAGE_DIR, exist_ok=True)
LOG_PATH = os.path.join(STORAGE_DIR, "posted_log.json")
QUEUE_PATH = os.path.join(STORAGE_DIR, "tweet_queue.json")
print(f"Storing data in: {STORAGE_DIR}")


Storing data in: /content/twitter_bot_data


## 7. Your tweet queue

This is the actual content the bot will post, in order. Edit this list with whatever you want to post - each entry is one tweet (keep them under 280 characters). Running this cell saves the list to `tweet_queue.json`; edit the list and re-run any time you want to add more.

If a queue file already exists (from a previous session), this adds your new tweets onto the end rather than wiping it out.

In [9]:
new_tweets = [
    "Just automated my Twitter posting with Python and the tweepy library. Small script, big time saver.",
    "Working on a side project this week - more updates soon. #buildinpublic",
    "Reminder: consistency beats intensity. Show up daily, even for 10 minutes.",
    "Python tip: f-strings make debugging so much cleaner than .format(). Small thing, huge quality of life boost.",
    "Currently exploring API integrations - there's something satisfying about two systems talking to each other correctly.",
]

if os.path.exists(QUEUE_PATH):
    with open(QUEUE_PATH) as f:
        queue = json.load(f)
else:
    queue = []

existing_texts = {item["text"] for item in queue}
for text in new_tweets:
    if text not in existing_texts:
        queue.append({"text": text, "posted": False})

with open(QUEUE_PATH, "w") as f:
    json.dump(queue, f, indent=2)

pending = sum(1 for t in queue if not t["posted"])
print(f"Queue has {len(queue)} tweet(s) total, {pending} still waiting to be posted.")


Queue has 5 tweet(s) total, 5 still waiting to be posted.


## 8. The actual posting function

Handles the dry-run case, talks to the API, and deals with the most common thing that goes wrong: rate limits. If Twitter tells us to slow down, this waits until the limit resets instead of just crashing.

In [10]:
def load_log():
    if os.path.exists(LOG_PATH):
        with open(LOG_PATH) as f:
            return json.load(f)
    return []


def save_log(log):
    with open(LOG_PATH, "w") as f:
        json.dump(log, f, indent=2)


def post_tweet(text: str) -> bool:
    # Posts one tweet. Returns True if it went out (or would have, in dry-run), False if it failed.
    timestamp = datetime.now().isoformat(timespec="seconds")

    if DRY_RUN:
        print(f"[DRY RUN] Would post at {timestamp}:\n  \"{text}\"\n")
        return True

    try:
        response = client.create_tweet(text=text)
        tweet_id = response.data["id"]
        print(f"Posted at {timestamp} -> https://twitter.com/i/web/status/{tweet_id}")
        return True

    except tweepy.TooManyRequests as e:
        reset_time = int(e.response.headers.get("x-rate-limit-reset", time.time() + 900))
        wait_seconds = max(reset_time - int(time.time()), 60)
        print(f"Rate limited - waiting {wait_seconds} seconds before trying again.")
        time.sleep(wait_seconds)
        return post_tweet(text)  # try again after waiting

    except tweepy.TweepyException as e:
        print(f"Failed to post tweet: {e}")
        return False


## 9. Post the next tweet in the queue (run once, manually)

Run this cell any time you just want to post the next pending tweet right now, without setting up the automatic scheduler below.

In [11]:
def post_next_in_queue():
    with open(QUEUE_PATH) as f:
        queue = json.load(f)

    for item in queue:
        if not item["posted"]:
            success = post_tweet(item["text"])
            if success:
                item["posted"] = True
                with open(QUEUE_PATH, "w") as f:
                    json.dump(queue, f, indent=2)

                log = load_log()
                log.append({"text": item["text"], "posted_at": datetime.now().isoformat(timespec="seconds")})
                save_log(log)
            return

    print("Queue is empty - nothing left to post. Add more tweets in the cell above.")


post_next_in_queue()


[DRY RUN] Would post at 2026-08-09T00:58:16:
  "Just automated my Twitter posting with Python and the tweepy library. Small script, big time saver."



## 10. Automatic scheduling loop

This is the "set it and forget it" part - it posts one tweet from the queue, waits a set number of minutes, then posts the next one, and repeats. Since a Colab cell has to keep running for this to work, it'll only keep going as long as this notebook tab stays open and connected (this is the main practical limit of running something like this from Colab instead of a proper server).

`max_posts` puts a hard ceiling on how many it'll send in one run, mostly as a safety net so a typo doesn't accidentally spam an entire queue at once.

In [12]:
def run_scheduler(interval_minutes=60, max_posts=5):
    posts_sent = 0
    print(f"Scheduler started - posting every {interval_minutes} minute(s), up to {max_posts} tweet(s) this run.")
    print("Interrupt the cell (stop button) any time to end the run early.\n")

    while posts_sent < max_posts:
        with open(QUEUE_PATH) as f:
            queue = json.load(f)

        if not any(not item["posted"] for item in queue):
            print("Queue is empty - stopping the scheduler.")
            break

        post_next_in_queue()
        posts_sent += 1

        if posts_sent < max_posts:
            next_time = (datetime.now() + timedelta(minutes=interval_minutes)).strftime("%H:%M:%S")
            print(f"Next post at approximately {next_time}. Sleeping...\n")
            time.sleep(interval_minutes * 60)

    print(f"\nDone for this run - sent {posts_sent} tweet(s).")


# Example: post every 2 minutes, up to 3 tweets, just to see it work in dry-run mode.
# For real use, something like interval_minutes=120 (every 2 hours) is more typical.
run_scheduler(interval_minutes=2, max_posts=3)


Scheduler started - posting every 2 minute(s), up to 3 tweet(s) this run.
Interrupt the cell (stop button) any time to end the run early.

[DRY RUN] Would post at 2026-08-09T00:58:16:
  "Working on a side project this week - more updates soon. #buildinpublic"

Next post at approximately 01:00:16. Sleeping...

[DRY RUN] Would post at 2026-08-09T01:00:16:
  "Reminder: consistency beats intensity. Show up daily, even for 10 minutes."

Next post at approximately 01:02:16. Sleeping...

[DRY RUN] Would post at 2026-08-09T01:02:16:
  "Python tip: f-strings make debugging so much cleaner than .format(). Small thing, huge quality of life boost."


Done for this run - sent 3 tweet(s).


## 11. (Optional) Auto-reply to mentions

A simple automation add-on: checks recent mentions of your account and replies to any it hasn't already responded to. Useful as a basic "thanks for the mention" auto-responder. Keep the reply list short and generic - this isn't meant to hold a real conversation, just acknowledge people automatically.

In [13]:
REPLY_TEMPLATES = [
    "Thanks for the mention!",
    "Appreciate you tagging me in this.",
    "Thanks for reaching out!",
]

def auto_reply_to_mentions(max_replies=5):
    if DRY_RUN:
        print("[DRY RUN] Would check mentions and reply - skipping since no real connection is set up.")
        return

    me = client.get_me().data
    mentions = client.get_users_mentions(id=me.id, max_results=max_replies)

    if not mentions.data:
        print("No new mentions found.")
        return

    log = load_log()
    replied_ids = {entry.get("replied_to_tweet_id") for entry in log if "replied_to_tweet_id" in entry}

    for tweet in mentions.data:
        if str(tweet.id) in replied_ids:
            continue
        reply_text = random.choice(REPLY_TEMPLATES)
        try:
            client.create_tweet(text=reply_text, in_reply_to_tweet_id=tweet.id)
            print(f"Replied to tweet {tweet.id}")
            log.append({"replied_to_tweet_id": str(tweet.id), "replied_at": datetime.now().isoformat(timespec="seconds")})
            save_log(log)
        except tweepy.TweepyException as e:
            print(f"Couldn't reply to {tweet.id}: {e}")


auto_reply_to_mentions()


[DRY RUN] Would check mentions and reply - skipping since no real connection is set up.


## Notes and honest limitations

- **This is not a 24/7 server.** The scheduling loop only runs as long as the Colab tab is open and the runtime hasn't disconnected. For genuinely round-the-clock posting, this same code would need to run on something always-on - a small VPS, a Raspberry Pi, GitHub Actions on a cron schedule, etc. Colab is great for building and testing this, less great as the permanent home for it.
- **Twitter/X API access levels change.** What you can do for free versus what needs a paid tier has shifted over time - check the developer portal for your account's current limits before assuming this will "just work."
- **Rate limits are real.** The retry logic here waits out a rate limit once it hits one, but if you're posting a lot in a short window you'll hit them more often - space posts out generously.
- **The auto-reply function is intentionally basic.** It's a fixed set of canned replies, not anything context-aware. Good as a starting point, not a real conversational bot.
